# Solutions · 14. Bagging, random forests and permutation importance

Worked solutions to the three exercises of [Lecture 14](../../notebooks/part5_ensembles/14_random_forests_bagging_and_feature_importance.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Single CART Tree vs. 25-Tree Random Forest Out-of-Sample Benchmark")
rng = np.random.default_rng(42)
N = 70
# 4 descriptors: x0 (Temperature, dominant), x1 (Pressure, moderate), x2 & x3 (pure irrelevant noise):
X_tr = rng.uniform(-2.0, 2.0, size=(N, 4))
y_tr_clean = 3.0 * np.sin(X_tr[:, 0]) + 1.5 * (X_tr[:, 1] ** 2)
y_tr = y_tr_clean + rng.normal(0.0, 0.45, size=N)

X_te = rng.uniform(-2.0, 2.0, size=(120, 4))
y_te_clean = 3.0 * np.sin(X_te[:, 0]) + 1.5 * (X_te[:, 1] ** 2)

single_tree = trees_ensembles.DecisionTreeRegressor(max_depth=5, min_samples_split=3).fit(X_tr, y_tr)
rf_model = trees_ensembles.RandomForestRegressor(n_estimators=25, max_depth=5, min_samples_split=3, seed=42).fit(X_tr, y_tr)

mse_single = float(np.mean((single_tree.predict(X_te) - y_te_clean) ** 2))
mse_rf = float(np.mean((rf_model.predict(X_te) - y_te_clean) ** 2))
print(f"Out-of-sample Test MSE against true physical surface: Single Tree = {mse_single:.4f} -> 25-Tree Random Forest = {mse_rf:.4f} ({100*(1-mse_rf/mse_single):.1f}% reduction)")
assert mse_rf < 0.75 * mse_single
heading("Demonstration 2: Equicorrelated Ensemble Covariance Identity & Tree Dispersion")
B, sig2, rho = 20, 2.0, 0.25
Sigma_ens = sig2 * (rho * np.ones((B, B)) + (1.0 - rho) * np.eye(B))
var_matrix = float(np.sum(Sigma_ens)) / (B ** 2)
var_formula = rho * sig2 + (1.0 - rho) * sig2 / B
print(f"Exact covariance quadratic form (1/B^2) 1^T Sigma 1 = {var_matrix:.6f} == Formula = {var_formula:.6f} (vs single tree = {sig2:.2f})")
assert np.isclose(var_matrix, var_formula, rtol=1e-15)

all_tree_preds = np.column_stack([t.predict(X_te) for t in rf_model.trees_])
mean_single_tree_mse = float(np.mean((all_tree_preds - y_te_clean[:, None]) ** 2))
print(f"Average MSE of the 25 individual bootstrap trees = {mean_single_tree_mse:.4f} vs Ensemble Mean MSE = {mse_rf:.4f}")
assert mse_rf < 0.55 * mean_single_tree_mse
heading("Demonstration 3: Breiman Permutation Feature Importance Descriptor Ranking")
imp = trees_ensembles.permutation_feature_importance(rf_model, X_te, y_te_clean, seed=42)
names = ["x0 (Temperature - strong 3*sin)", "x1 (Pressure - moderate 1.5*x^2)", "x2 (Inactive noise #1)", "x3 (Inactive noise #2)"]
for j, name in enumerate(names):
    print(f"Feature {name:33s} | Permutation Delta-MSE = {imp[j]:.4f}")

assert imp[0] > imp[1] > 0.5
assert imp[2] < 0.15 * imp[1] and imp[3] < 0.15 * imp[1]
heading("Demonstration 4: Ensemble Size B Convergence Sweep")
print(f"{'Forest Size B':>13s} | {'Test MSE vs True Surface':>24s} | {'Ensemble Std Across Trees':>25s}")
print("-" * 68)
for b_trees in [1, 3, 5, 10, 20, 35]:
    rf_b = trees_ensembles.RandomForestRegressor(n_estimators=b_trees, max_depth=5, seed=42).fit(X_tr, y_tr)
    mse_b = float(np.mean((rf_b.predict(X_te) - y_te_clean) ** 2))
    t_mat = np.column_stack([t.predict(X_te) for t in rf_b.trees_])
    std_b = float(np.mean(np.std(t_mat, axis=1)))
    print(f"{b_trees:13d} | {mse_b:24.4f} | {std_b:25.4f}")
heading("Inside the Algorithm: By-Hand Ensemble Averaging & Krogh-Vedelsby Diversity Identity")
pred_rf_hand = np.mean(all_tree_preds, axis=1)
pred_rf_lib = rf_model.predict(X_te)
diff_rf = float(np.max(np.abs(pred_rf_hand - pred_rf_lib)))
diversity_term = float(np.mean((all_tree_preds - pred_rf_hand[:, None]) ** 2))
kv_lhs = mean_single_tree_mse - mse_rf
print(f"by hand vs library max discrepancy : {diff_rf:.2e}")
print(f"Krogh-Vedelsby Identity: Mean(MSE_b) - MSE_RF = {kv_lhs:.6f} == Ensemble Variance (Diversity) = {diversity_term:.6f}")
assert diff_rf < 1e-15 and np.isclose(kv_lhs, diversity_term, rtol=1e-14)
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Prove and verify numerically the **bootstrap out-of-bag (OOB) asymptotic law**: when drawing $N$ samples with replacement from $N$ items, the probability that any given observation $i$ is omitted from a bootstrap draw is $(1 - 1/N)^N \to e^{-1} \approx 0.367879$. Run 500 bootstrap draws for $N = 100$ and verify that the empirical OOB fraction lies within $0.368 \pm 0.015$.

In [2]:
heading("Exercise 1: Bootstrap Out-of-Bag (OOB) 1/e Asymptotic Law Verification")
rng_oob = np.random.default_rng(2026)
N_oob = 100
exact_prob_oob = (1.0 - 1.0 / N_oob) ** N_oob
oob_fracs = [1.0 - len(np.unique(rng_oob.integers(0, N_oob, size=N_oob))) / N_oob for _ in range(500)]
emp_oob = float(np.mean(oob_fracs))
print(f"Exact finite-N OOB probability (1 - 1/N)^N = {exact_prob_oob:.6f} | 1/e = {np.exp(-1.0):.6f} | 500-trial Empirical = {emp_oob:.6f}")
assert abs(emp_oob - exact_prob_oob) < 0.01


Exercise 1: Bootstrap Out-of-Bag (OOB) 1/e Asymptotic Law Verification
Exact finite-N OOB probability (1 - 1/N)^N = 0.366032 | 1/e = 0.367879 | 500-trial Empirical = 0.367100


**Discussion.** Roughly $36.8\%$ of the training data is left out of each bootstrap tree, providing a built-in Out-of-Bag cross-validation set without needing a separate validation split.

---

## Exercise 2

**Problem.** Audit the **collinear descriptor dilution effect** on permutation feature importance: append a near-duplicate copy of Temperature $x_4 = x_0 + \mathcal{N}(0, 0.01^2)$ to `X_tr` and `X_te`, refit the random forest, and show that splitting the physical signal across two redundant collinear temperature columns dilutes the individual permutation importance of $x_0$.

In [3]:
heading("Exercise 2: Collinear Feature Importance Dilution Audit")
X_tr_col = np.column_stack([X_tr, X_tr[:, 0] + rng.normal(0.0, 0.15, size=N)])
X_te_col = np.column_stack([X_te, X_te[:, 0] + rng.normal(0.0, 0.15, size=120)])
rf_col = trees_ensembles.RandomForestRegressor(n_estimators=25, max_depth=5, seed=42).fit(X_tr_col, y_tr)
imp_col = trees_ensembles.permutation_feature_importance(rf_col, X_te_col, y_te_clean, seed=42)
print(f"Original x0 importance (no duplicate)       : {imp[0]:.4f}")
print(f"Duplicated [x0, x4_copy] individual importances: x0 = {imp_col[0]:.4f}, x4_copy = {imp_col[4]:.4f} (Sum = {imp_col[0]+imp_col[4]:.4f})")
assert imp_col[0] < 0.85 * imp[0] and imp_col[4] > 0.50


Exercise 2: Collinear Feature Importance Dilution Audit


Original x0 importance (no duplicate)       : 7.3553
Duplicated [x0, x4_copy] individual importances: x0 = 1.1173, x4_copy = 4.5248 (Sum = 5.6421)


**Discussion.** With a near-copy of $x_0$ in the data, the trees can split on either column, and shuffling one of them leaves the other intact. The importance of the original falls from $7.36$ to $1.12$, the copy receives $4.52$, and the two together ($5.64$) are less than the original alone. The split between the pair is uneven and depends on the run. Permutation importance measures what the model loses when one column is destroyed, not how much the response depends on the quantity behind it; correlated descriptors should be shuffled together.

---

## Exercise 3

**Problem.** Implement an **out-of-bag (OOB) random forest evaluator** `fit_rf_with_oob_score(X: np.ndarray, y: np.ndarray, n_estimators: int = 30, max_depth: int = 5, seed: int = 42) -> dict` that records each tree's bootstrap indices, predicts each training point $i$ using *only* the trees for which $i \notin \mathcal{D}_b^*$, and returns the OOB MSE. Verify on `(X_tr, y_tr)` that the OOB MSE estimates the true noisy test MSE without touching `X_te`.

In [4]:
heading("Exercise 3: Implement Out-of-Bag (OOB) Validation Estimator From Scratch")
def fit_rf_with_oob_score(X: np.ndarray, y: np.ndarray, n_estimators: int = 30, max_depth: int = 5, seed: int = 42) -> dict:
    """Fit a bagged CART forest while accumulating Out-of-Bag (OOB) predictions for every training sample."""
    Xa = np.asarray(X, dtype=float)
    ya = np.asarray(y, dtype=float).ravel()
    n_s = len(ya)
    rng_b = np.random.default_rng(seed)
    oob_sum = np.zeros(n_s)
    oob_cnt = np.zeros(n_s)
    for _ in range(n_estimators):
        b_idx = rng_b.integers(0, n_s, size=n_s)
        t_b = trees_ensembles.DecisionTreeRegressor(max_depth=max_depth, min_samples_split=3).fit(Xa[b_idx], ya[b_idx])
        oob_mask = np.ones(n_s, dtype=bool)
        oob_mask[b_idx] = False
        if np.any(oob_mask):
            oob_sum[oob_mask] += t_b.predict(Xa[oob_mask])
            oob_cnt[oob_mask] += 1.0
    valid = oob_cnt > 0
    oob_pred = np.where(valid, oob_sum / np.maximum(oob_cnt, 1.0), np.mean(ya))
    return {"oob_mse": float(np.mean((oob_pred[valid] - ya[valid]) ** 2)), "oob_coverage": float(np.mean(valid))}

oob_res = fit_rf_with_oob_score(X_tr, y_tr, n_estimators=30, max_depth=5)
print(f"OOB Sample Coverage = {oob_res['oob_coverage']*100:.1f}% | OOB Validation MSE = {oob_res['oob_mse']:.4f}")
assert oob_res["oob_coverage"] == 1.0 and 0.5 < oob_res["oob_mse"] < 2.5


Exercise 3: Implement Out-of-Bag (OOB) Validation Estimator From Scratch


OOB Sample Coverage = 100.0% | OOB Validation MSE = 1.6888


**Discussion.** Because $(1 - e^{-1})^{30} < 10^{-6}$, every training point is out-of-bag in roughly $11$ of $30$ trees, which gives an estimate of the test error without a separate validation set.